<a href="https://colab.research.google.com/github/janahendy30/One-Model-Many-Dialects/blob/main/data_preparation_and_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q "transformers<5" accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/thesis"
RAW, PROC = f"{BASE}/data/raw", f"{BASE}/data/processed"
os.makedirs(PROC, exist_ok=True)

print(os.listdir(RAW))
print(os.listdir(f"{RAW}/MADAR_Corpus"))

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

import torch, math
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "inceptionai/jais-family-1p3b"
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto", trust_remote_code=True)

In [ ]:
inputs = tok("عاصمة مصر هي", return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=15, do_sample=False, use_cache=False)
print(tok.decode(out[0], skip_special_tokens=True))

In [ ]:
text = "تعد القاهرة عاصمة جمهورية مصر العربية وأكبر مدنها، وهي من أهم المراكز الثقافية والاقتصادية في العالم العربي. تقع المدينة على ضفاف نهر النيل، ويعود تاريخها إلى آلاف السنين."
enc = tok(text, return_tensors="pt").to(model.device)
with torch.no_grad():
    loss = model(**enc, labels=enc["input_ids"]).loss
print("tokens:", enc["input_ids"].shape[1], "perplexity:", math.exp(loss.item()))

In [ ]:
print(tok.bos_token, tok.eos_token)
print(tok("أريد ركوب الجمال").input_ids)
print(tok.convert_ids_to_tokens(tok("أريد ركوب الجمال").input_ids))

In [ ]:
print(model)

In [ ]:
print(os.listdir(f"{RAW}/MADAR_Corpus"))

In [ ]:
import os, re, json, random
import pandas as pd

BASE = "/content/drive/MyDrive/thesis"
RAW, PROC = f"{BASE}/data/raw", f"{BASE}/data/processed"
os.makedirs(PROC, exist_ok=True)

# ---------- cleaning ----------
DIAC = re.compile(r"[\u064B-\u0652\u0670\u0640]")   # diacritics + tatweel
AR = re.compile(r"[\u0600-\u06FF]")

def clean(t):
    t = str(t).replace("\ufeff", "")                 # hidden BOM
    t = re.sub(r"http\S+|www\.\S+", " ", t)          # URLs
    t = re.sub(r"@\w+", " ", t)                      # mentions
    t = t.replace("#", " ")                          # hashtag symbol
    t = DIAC.sub("", t)                              # same spelling style for all corpora
    return re.sub(r"\s+", " ", t).strip()

def arabic_ratio(t):
    letters = [c for c in t if c.isalpha()]
    return sum(bool(AR.match(c)) for c in letters) / len(letters) if letters else 0

def key(t):   # for duplicate and overlap checks (ignores punctuation and spacing)
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", t)).strip()

# ---------- MADAR parallel test set (official CAMeL release) ----------
MADAR_DIR = f"{RAW}/MADAR_Corpus"
cities = {"msa": "MSA", "saudi_riy": "Riyadh", "saudi_jed": "Jeddah",
          "egyptian_cai": "Cairo", "moroccan_rab": "Rabat", "moroccan_fes": "Fes"}

tables = {}
for col, city in cities.items():
    df = pd.read_csv(f"{MADAR_DIR}/MADAR.corpus.{city}.tsv", sep="\t", quoting=3)
    tables[col] = df.set_index("sentID.BTEC")["sent"]

common_ids = sorted(set.intersection(*[set(s.index) for s in tables.values()]))
madar = pd.DataFrame({"id": common_ids,
                      **{col: [clean(s[i]) for i in common_ids] for col, s in tables.items()}})
madar.to_json(f"{PROC}/madar_parallel_test.jsonl", orient="records",
              lines=True, force_ascii=False)
madar_keys = {key(x) for col in cities for x in madar[col]}
print("MADAR parallel test:", madar.shape)          # expect (2000, 7)

# ---------- training corpora ----------
def prepare(texts, min_words=2):
    out, seen = [], set()
    for t in texts:
        if pd.isna(t):
            continue
        t = clean(t)
        k = key(t)
        if len(t.split()) < min_words or arabic_ratio(t) < 0.6:
            continue
        if k in seen or k in madar_keys:             # drop duplicates and MADAR overlap
            continue
        seen.add(k); out.append(t)
    return out

saudi    = prepare(open(f"{RAW}/SDC.txt", encoding="utf-8").read().splitlines())
egyptian = prepare(pd.read_excel(f"{RAW}/3.TaghreedT EDC.xlsx")["Text"])
moroccan = prepare(pd.read_csv(f"{RAW}/sentences Moroccan.csv")["darija_ar"])

print()
for name, d in [("saudi", saudi), ("egyptian", egyptian), ("moroccan", moroccan)]:
    print(f"{name}: {len(d)} texts after cleaning, {sum(len(t.split()) for t in d)} words")

# ---------- split: test 1,000 / dev 500 / rest train, per dialect ----------
random.seed(42)
TEST_SIZE, DEV_SIZE = 1000, 500
splits = {}
print()
for name, d in [("saudi", saudi), ("egyptian", egyptian), ("moroccan", moroccan)]:
    d = d[:]
    random.shuffle(d)
    splits[name] = {
        "test":  d[:TEST_SIZE],
        "dev":   d[TEST_SIZE:TEST_SIZE + DEV_SIZE],
        "train": d[TEST_SIZE + DEV_SIZE:],
    }
    for part, texts in splits[name].items():
        with open(f"{PROC}/{name}_{part}.jsonl", "w", encoding="utf-8") as f:
            for t in texts:
                f.write(json.dumps({"text": t, "dialect": name}, ensure_ascii=False) + "\n")
    print(f"{name}: train {len(splits[name]['train'])}, "
          f"dev {len(splits[name]['dev'])}, test {len(splits[name]['test'])}")

print("\nSaved files:", sorted(os.listdir(PROC)))

In [ ]:
for name in ["saudi", "egyptian", "moroccan"]:
    print(f"\n=== {name} ===")
    for t in random.sample(splits[name]["train"], 15):
        print("-", t[:150])

print("\n=== MADAR (first 3 rows) ===")
print(madar.head(3).to_string())

In [ ]:
#Token matching
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("inceptionai/jais-family-1p3b", trust_remote_code=True)

def load(name):
    with open(f"{PROC}/{name}_train.jsonl", encoding="utf-8") as f:
        return [json.loads(l)["text"] for l in f]

train = {d: load(d) for d in ["saudi", "egyptian", "moroccan"]}
counts = {d: [len(tok(t).input_ids) for t in texts] for d, texts in train.items()}

for d in train:
    print(f"{d}: {len(train[d])} texts, {sum(counts[d])} tokens")

budget = min(sum(c) for c in counts.values())          # the smallest dialect sets the budget
print("\nToken budget per dialect:", budget)

# the training files are already shuffled (seed 42), so taking texts in order is a random sample
for d in train:
    kept, total = [], 0
    for t, n in zip(train[d], counts[d]):
        if total + n > budget:
            break
        kept.append(t); total += n
    with open(f"{PROC}/{d}_train_matched.jsonl", "w", encoding="utf-8") as f:
        for t in kept:
            f.write(json.dumps({"text": t, "dialect": d}, ensure_ascii=False) + "\n")
    print(f"{d}: kept {len(kept)} texts, {total} tokens")

In [ ]:
import os, re, json, random
import pandas as pd

alm_raw = pd.read_csv(f"{RAW}/almeman.csv.csv", encoding="utf-8-sig", dtype=str)
print("Raw:", alm_raw.shape, alm_raw.columns.tolist())

LABEL = re.compile(r"^\s*-?\s*(Saudi|Egyptian|Iraqi|Levantine|Moroccan)\s*:\s*", re.I)

alm = pd.DataFrame({
    "msa":      alm_raw["Modern Standard Arabic"],
    "saudi":    alm_raw["Saudi Dialect"],
    "egyptian": alm_raw["Egyptian Dialect"],
    "moroccan": alm_raw["Moroccan Dialect"],
}).dropna()

for c in alm.columns:
    alm[c] = alm[c].map(lambda x: clean(LABEL.sub("", str(x))))   # remove "- Saudi:" etc., then clean

# keep only proper Arabic rows in every column
alm = alm[alm.apply(lambda r: all(arabic_ratio(r[c]) >= 0.6 for c in alm.columns), axis=1)]

# remove anything that also appears in the training data (keeps the test unseen)
train_keys = set()
for d in ["saudi", "egyptian", "moroccan"]:
    with open(f"{PROC}/{d}_train_matched.jsonl", encoding="utf-8") as f:
        train_keys |= {key(json.loads(l)["text"]) for l in f}
before = len(alm)
alm = alm[~alm.apply(lambda r: any(key(r[c]) in train_keys for c in alm.columns), axis=1)]
print("Removed for overlap with training:", before - len(alm))
same_as_msa = alm.apply(lambda r: any(key(r[c]) == key(r["msa"]) # to remove rows where dialects are identical to MSA
                                      for c in ["saudi", "egyptian", "moroccan"]), axis=1)
print("Removed rows where a dialect is identical to MSA:", same_as_msa.sum())
alm = alm[~same_as_msa]
alm = alm.drop_duplicates(subset="msa").sample(n=2000, random_state=42).reset_index(drop=True)
alm.to_json(f"{PROC}/almeman_parallel_test.jsonl", orient="records", lines=True, force_ascii=False)
print("Almeman test:", alm.shape)
print(alm.head(3).to_string())

In [ ]:
import datasets
try:
    mmlu = datasets.load_dataset("MBZUAI/ArabicMMLU")
except Exception as e:
    print("Needs a config name:", e)
    mmlu = datasets.load_dataset("MBZUAI/ArabicMMLU", "All")

print(mmlu)
for split in mmlu:
    print(split, mmlu[split].column_names)
first = list(mmlu.keys())[0]
print(mmlu[first][0])

mmlu.save_to_disk(f"{RAW}/arabicmmlu")

In [ ]:
test_split = "test" if "test" in mmlu else first
df = mmlu[test_split].to_pandas()

needed = ["Question", "Answer Key"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Columns not found: {missing}. Available: {df.columns.tolist()}")

group_col = "Group" if "Group" in df.columns else "Subject"
print("Questions per", group_col, ":\n", df[group_col].value_counts())

PER_GROUP = 300
subset = (df.groupby(group_col, group_keys=False)
            .apply(lambda g: g.sample(n=min(PER_GROUP, len(g)), random_state=42))
            .reset_index(drop=True))

subset.to_json(f"{PROC}/arabicmmlu_test_subset.jsonl", orient="records",
               lines=True, force_ascii=False)
print("\nArabicMMLU subset:", subset.shape)
print(subset[group_col].value_counts())

# few-shot examples, kept separate from the test questions
if "dev" in mmlu:
    mmlu["dev"].to_pandas().to_json(f"{PROC}/arabicmmlu_dev_fewshot.jsonl",
                                    orient="records", lines=True, force_ascii=False)
    print("Few-shot dev examples saved:", len(mmlu["dev"]))

In [ ]:
import os, json, math, time
import pandas as pd
import torch
import torch.nn.functional as F

RESULTS = f"{BASE}/results"
os.makedirs(RESULTS, exist_ok=True)

tok.pad_token = tok.eos_token
tok.padding_side = "right"
model.eval()
print("GPU memory used:", round(torch.cuda.memory_allocated() / 1e9, 1), "GB")

In [ ]:
#3a)Perplexity baseline
@torch.no_grad()
def evaluate_texts(model, tok, texts, batch_size=16, max_len=512):
    """Returns perplexity and bits-per-character over a list of texts."""
    total_nll, total_tokens, total_chars = 0.0, 0, 0
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        enc = tok([tok.bos_token + t for t in batch], return_tensors="pt",
                  padding=True, truncation=True, max_length=max_len).to(model.device)

        logits = model(input_ids=enc.input_ids, attention_mask=enc.attention_mask,
                       use_cache=False).logits

        # predict token t+1 from tokens up to t
        shift_logits = logits[:, :-1, :].float()
        shift_labels = enc.input_ids[:, 1:]
        shift_mask   = enc.attention_mask[:, 1:].bool()      # ignore padding

        nll = F.cross_entropy(shift_logits.reshape(-1, shift_logits.size(-1)),
                              shift_labels.reshape(-1), reduction="none")
        nll = nll.view(shift_labels.shape)[shift_mask]

        total_nll    += nll.sum().item()
        total_tokens += shift_mask.sum().item()
        total_chars  += sum(len(t) for t in batch)

    return {"ppl": math.exp(total_nll / total_tokens),
            "bpc": total_nll / (math.log(2) * total_chars),
            "n_texts": len(texts), "n_tokens": total_tokens}

In [ ]:
text = "تعد القاهرة عاصمة جمهورية مصر العربية وأكبر مدنها، وهي من أهم المراكز الثقافية والاقتصادية في العالم العربي. تقع المدينة على ضفاف نهر النيل، ويعود تاريخها إلى آلاف السنين."
print(evaluate_texts(model, tok, [text]))

In [ ]:
text = "تعد القاهرة عاصمة جمهورية مصر العربية وأكبر مدنها، وهي من أهم المراكز الثقافية والاقتصادية في العالم العربي."

# Hugging Face's own loss
enc = tok(tok.bos_token + text, return_tensors="pt").to(model.device)
with torch.no_grad():
    hf_loss = model(**enc, labels=enc.input_ids, use_cache=False).loss.item()
print("Hugging Face perplexity:", math.exp(hf_loss))

# your function
print("Your function:          ", evaluate_texts(model, tok, [text])["ppl"])

In [ ]:
text = "تعد القاهرة عاصمة جمهورية مصر العربية وأكبر مدنها، وهي من أهم المراكز الثقافية والاقتصادية في العالم العربي."

# 1. are the inputs identical?
a = tok(tok.bos_token + text, return_tensors="pt").to(model.device)
b = tok([tok.bos_token + text], return_tensors="pt", padding=True,
        truncation=True, max_length=512).to(model.device)
print("same input ids:", torch.equal(a.input_ids, b.input_ids), "| tokens:", a.input_ids.shape[1])
print("model dtype:", next(model.parameters()).dtype)

with torch.no_grad():
    # 2. Hugging Face's own loss, and its logits
    out_hf = model(**a, labels=a.input_ids, use_cache=False)
    # 3. a second forward pass the way my function does it
    out_mine = model(input_ids=b.input_ids, attention_mask=b.attention_mask, use_cache=False)

# 4. do the two forward passes give the same logits?
print("max logit difference:", (out_hf.logits.float() - out_mine.logits.float()).abs().max().item())

# 5. standard cross-entropy computed on Hugging Face's own logits
lg = out_hf.logits[:, :-1, :].float()
lb = a.input_ids[:, 1:]
manual = F.cross_entropy(lg.reshape(-1, lg.size(-1)), lb.reshape(-1))
print("HF loss:", out_hf.loss.item(), "| manual loss on HF logits:", manual.item())

In [ ]:
print(a.keys())

In [ ]:
with torch.no_grad():
    out_clean = model(input_ids=a.input_ids, attention_mask=a.attention_mask,
                      labels=a.input_ids, use_cache=False)
print("HF perplexity without token_type_ids:", math.exp(out_clean.loss.item()))
print("my function:                       ", evaluate_texts(model, tok, [text])["ppl"])

In [ ]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f]

def evaluate_all(model, tok, condition="base", method="none", trained_on="none", seed=0):
    rows = []
    def record(test_set, variety, texts):
        start = time.time()
        r = evaluate_texts(model, tok, texts)
        rows.append({"model": MODEL, "condition": condition, "method": method,
                     "trained_on": trained_on, "seed": seed,
                     "test_set": test_set, "variety": variety, **r})
        print(f"{test_set:8s} {variety:13s} ppl={r['ppl']:8.2f}  bpc={r['bpc']:.3f}  "
              f"({r['n_texts']} texts, {time.time() - start:.0f}s)")

    # dialect held-out test sets
    for d in ["saudi", "egyptian", "moroccan"]:
        record("heldout", d, [x["text"] for x in read_jsonl(f"{PROC}/{d}_test.jsonl")])

    # MADAR (main parallel set)
    madar = pd.read_json(f"{PROC}/madar_parallel_test.jsonl", lines=True)
    for col in ["msa", "saudi_riy", "saudi_jed", "egyptian_cai", "moroccan_rab", "moroccan_fes"]:
        record("madar", col, madar[col].tolist())

    # Almeman (supporting parallel set)
    alm = pd.read_json(f"{PROC}/almeman_parallel_test.jsonl", lines=True)
    for col in ["msa", "saudi", "egyptian", "moroccan"]:
        record("almeman", col, alm[col].tolist())

    return pd.DataFrame(rows)

baseline = evaluate_all(model, tok)
baseline.to_csv(f"{RESULTS}/baseline_perplexity.csv", index=False)
print("\nSaved:", f"{RESULTS}/baseline_perplexity.csv")

In [ ]:
#3b) MMLU Accuracy
import numpy as np

LEVEL_EN = {"Primary": "primary school", "Middle": "middle school", "High": "high school",
            "Univ": "university", "Prof": "professional"}
LETTERS = ["A", "B", "C", "D", "E"]

def options_of(row):
    opts = []
    for i in range(1, 6):
        v = row.get(f"Option {i}")
        if v is None or pd.isna(v):
            break
        opts.append(str(v))
    return opts

def build_prompt(row):
    level = "" if pd.isna(row["Level"]) else " for " + LEVEL_EN[row["Level"]]
    country = "" if pd.isna(row["Country"]) else " in " + row["Country"]
    meta = f"{row['Subject']} question{level}{country}"
    question = row["Question"] if pd.isna(row["Context"]) else f"{row['Context']}\n\n{row['Question']}"
    opts = "\n".join(f"{LETTERS[i]}. {o}" for i, o in enumerate(options_of(row)))
    return f"This is a {meta}. Select the correct answer!\n\nQuestion: {question}\n{opts}\n\nAnswer: "

def build_fewshot_prompt(row, dev_df, k):
    if k == 0:
        return build_prompt(row)
    shots = dev_df[dev_df["Subject"] == row["Subject"]].head(k)
    prefix = "".join(build_prompt(s) + s["Answer Key"] + "\n\n" for _, s in shots.iterrows())
    return prefix + build_prompt(row)

subset = pd.read_json(f"{PROC}/arabicmmlu_test_subset.jsonl", lines=True)
dev_df = pd.read_json(f"{PROC}/arabicmmlu_dev_fewshot.jsonl", lines=True)
print(build_prompt(subset.iloc[0]))

In [ ]:
# exactly the same are their code
LETTER_IDS = [tok.encode(l)[-1] for l in LETTERS]      # same as their code

@torch.no_grad()
def predict(model, tok, prompt, n_options, max_len=2048):
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=max_len,
              return_token_type_ids=False).to(model.device)
    logits = model(input_ids=enc.input_ids, attention_mask=enc.attention_mask,
                   use_cache=False).logits[0, -1, :]
    scores = logits[LETTER_IDS[:n_options]].float().cpu().numpy()
    conf = np.exp(scores - scores.max()); conf /= conf.sum()
    return int(np.argmax(scores)), conf

def evaluate_mmlu(model, tok, subset, dev_df, k=0):
    preds, rows = [], []
    for _, row in subset.iterrows():
        n = len(options_of(row))
        pred, conf = predict(model, tok, build_fewshot_prompt(row, dev_df, k), n)
        gold = LETTERS.index(row["Answer Key"])
        rows.append({"Group": row["Group"], "correct": pred == gold,
                     "chance": 1 / n, "confidence": float(conf.max())})
    df = pd.DataFrame(rows)
    summary = {"overall": df["correct"].mean() * 100,
               "chance": df["chance"].mean() * 100,
               **{g: grp["correct"].mean() * 100 for g, grp in df.groupby("Group")}}
    return summary, df

In [ ]:
mmlu_rows = []
for k in [0, 3]:
    start = time.time()
    summary, details = evaluate_mmlu(model, tok, subset, dev_df, k=k)
    details.to_csv(f"{RESULTS}/baseline_mmlu_{k}shot_details.csv", index=False)
    mmlu_rows.append({"model": MODEL, "condition": "base", "shots": k, **summary})
    print(f"{k}-shot: overall {summary['overall']:.1f}%  (chance {summary['chance']:.1f}%)  "
          f"[{time.time() - start:.0f}s]")
    for g in ["Humanities", "Language", "Other", "STEM", "Social Science"]:
        print(f"   {g:15s} {summary[g]:.1f}%")

pd.DataFrame(mmlu_rows).to_csv(f"{RESULTS}/baseline_mmlu.csv", index=False)